In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
from scipy.spatial.distance import cdist
from scipy import stats
import matplotlib.pyplot as plt
import pymannkendall as mk
import gc 


path1=' '

in_path=path1+" "

som_grids = {'grid3x4': (3, 4),}   #  'grid3x3': (3, 3), 'grid4x4': (4, 4),
grid_name='grid3x4'
grid_size=(3, 4)
days=(3,4,5,)# LDE duration criterion 
thresholds=(0.5,0.75,0.9,)# Euclidean distance percentile thresholds

In [ ]:
gph_anom=xr.open_dataarray("uyr_gph_anom_1979-2022.nc")
gph_anom['time'] = pd.to_datetime(gph_anom.time.values)

def get_season_indices(data_array):
    month = data_array.time.dt.month
    
    seasons = {
        'Spring': [3, 4, 5],
        'Summer': [6, 7, 8],
        'Autumn': [9, 10, 11],
        'Winter': [12, 1, 2]
    }
    
    season_ind = {}
    for season, months in seasons.items():
        
        mask = month.isin(months)
        
        season_ind[season] = np.where(mask)[0]
        
        print(f"{season}: {len(season_ind[season])} days")
        
    return season_ind

season_ind = get_season_indices(gph_anom)

for season in season_ind:
    print(f"{season} : {season_ind[season]}")

gc.collect()

Spring: 4048 days
Summer: 4048 days
Autumn: 4004 days
Winter: 3960 days
Spring : [   59    60    61 ... 15843 15844 15845]
Summer : [  151   152   153 ... 15935 15936 15937]
Autumn : [  243   244   245 ... 16026 16027 16028]
Winter : [    0     1     2 ... 16057 16058 16059]


103

In [ ]:
for day in days:
    for threshold in thresholds:
        print('dur_thres:',day,'per_thres:',threshold)
        #Identify LDE and WWE, store the nodes before and after transformation
        #for grid_name, grid_size in som_grids.items():    
        node_len=grid_size[0]*grid_size[1]
        print(grid_name)
        weights=pd.read_csv(in_path+f"som_results/gph_{grid_name}_total_weights.csv").values

        #Calculate the Euclidean distance matrix
        dist_matrix = cdist(weights, weights, metric='euclidean')

        matrix = pd.DataFrame(dist_matrix) 
        matrix.to_excel(in_path+f"som_results/gph_{grid_name}_total_distance.xlsx", index=False)
        classif=pd.read_csv(in_path+f"som_results/gph_{grid_name}_total_classif.csv").values.flatten()   #(16060,)
        diff_arr = np.diff(classif) #(16059,)

        node_ind=np.where(diff_arr == 0)[0]

        one_series=np.diff(node_ind)
        one_series[one_series!=1]=0

        e_start=node_ind[np.where((np.diff(one_series)== 1)&(one_series[1:]==1))[0]+1]  
        e_end=node_ind[np.where(np.diff(one_series)==-1)[0]+1]+1

        ind_end=np.where(node_ind==(e_start[-1]))[0]
        ind_start=np.where(node_ind==(e_end[0]-1))[0]

        if np.all(one_series[:ind_start[0]] == 1):
            e_start=np.insert(e_start,0, node_ind[0])
        if np.all(one_series[ind_end[0]:] == 1):
            e_end=np.insert(e_end,len(e_end), node_ind[-1])

        dur=e_end-e_start
        lde_ind=np.where(dur>=day)[0]
        lde_s=e_start[lde_ind]
        lde_e=e_end[lde_ind]
        print('lde_num', len(lde_e))
        node_c=[]
        node_n=[]
        for i in lde_e:
            node_c.append(classif[i])
            node_n.append(classif[i+2])

        distance=dist_matrix

        eu_dis=[]
        for i in range(len(node_c)):
            eu_dis.append(distance[(node_c[i]-1),(node_n[i]-1)])
        quan_dis=np.quantile(np.array(eu_dis),threshold)
        print('quan_dis:', quan_dis)

        n_trans=np.vstack((node_c,node_n,lde_s,lde_e, eu_dis)).T   #current node, next node, lde start, lde end

        trans_df = pd.DataFrame(n_trans, columns=['node_c', 'node_n', 'lde_s', 'lde_e', 'eu_dis'])
        group = trans_df.groupby('node_c').agg({'node_n': list, 'lde_s': list, 'lde_e': list, 'eu_dis': list})
        #print(group)

        #total
        #Changes in the frequencies of lde and wwe for each node
        years=np.arange(1979,2023)

        all_eu_dis = []
        for index, row in group.iterrows():
            all_eu_dis.extend(row['eu_dis'])

        quan_dis=np.quantile(np.array(all_eu_dis),threshold)

        #columns = ['Node', 'slope1', 'intercept1', 'z_value1', 'p_value_mk1', 'slope2', 'intercept2', 'z_value2', 'p_value_mk2']
        #results_df = pd.DataFrame(columns=columns)

        fig3, axs = plt.subplots(grid_size[0], grid_size[1],figsize=(18, 12))
        for i in range(grid_size[0]):
            for j in range(grid_size[1]):

                ax=axs[i,j]

                node=i*grid_size[1]+j+1

                e_n = []
                e_eu_dis = []

                if node in group.index:
                    e_n = group.loc[node, 'lde_s']
                    e_eu_dis = group.loc[node, 'eu_dis']

                    year=np.array(e_n)//365+1979

                    e_year=np.vstack((year,e_n,e_eu_dis)).T 
                    lde_year_df=pd.DataFrame(e_year, columns=['years', 'e_n','e_eu_dis'])

                    lde_year=lde_year_df.groupby('years').agg({'e_n': list})
                    lde_year['num'] = lde_year['e_n'].apply(len)

                    num1=np.zeros(44)

                    for y in range(len(np.array(lde_year.index))):
                        year=int(np.array(lde_year.index)[y])
                        num1[(year-1979)]=np.array(lde_year['num'])[y]
                        
                    slope1, intercept1, r_value1, p_value1, std_err1 = stats.linregress(years, num1)
                    y_trend1 = slope1 * years + intercept1

                    result1 = mk.original_test(num1)

                    wwe_year_df = lde_year_df[lde_year_df['e_eu_dis'] > quan_dis]

                    ax.plot(years,num1,color='black', alpha=0.6, linewidth=1)

                    if result1.p<0.05:
                        if result1.p<0.01:
                            ax.annotate(f'{slope1:.3f}**', xy=(0.35, 0.88), xycoords='axes fraction', horizontalalignment='left', fontsize=18, color='black', alpha=0.7)
                        else:
                            ax.annotate(f'{slope1:.3f}*', xy=(0.35, 0.88), xycoords='axes fraction', horizontalalignment='left', fontsize=18, color='black', alpha=0.7)
                        ax.plot(years,y_trend1,linewidth=2.5, color='black', alpha=0.6)
                    else:
                        ax.annotate(f'{slope1:.3f}', xy=(0.35, 0.88), xycoords='axes fraction', horizontalalignment='left', fontsize=18, color='black', alpha=0.7)

                    if not wwe_year_df.empty:
                        wwe_year=wwe_year_df.groupby('years').agg({'e_n': list})
                        wwe_year['num'] = wwe_year['e_n'].apply(len)

                        num2=np.zeros(44)

                        for y in range(len(np.array(wwe_year.index))):
                            year=int(np.array(wwe_year.index)[y])
                            num2[(year-1979)]=np.array(wwe_year['num'])[y]
                            
                        slope2, intercept2, r_value2, p_value2, std_err2 = stats.linregress(years, num2)
                        y_trend2 = slope2 * years + intercept2

                        result2 = mk.original_test(num2)

                        ax.plot(years,num2,color="C0", alpha=0.7, linewidth=1)

                        if result2.p<0.05:
                            if result2.p<0.01:
                                ax.annotate(f'{slope2:.3f}**', xy=(0.35, 0.78), xycoords='axes fraction', horizontalalignment='left', fontsize=18, color='C0')
                            else:
                                ax.annotate(f'{slope2:.3f}*', xy=(0.35, 0.78), xycoords='axes fraction', horizontalalignment='left', fontsize=18, color='C0')
                            ax.plot(years,y_trend2,linewidth=2.5, color='C0')
                        else:
                            ax.annotate(f'{slope2:.3f}', xy=(0.35, 0.78), xycoords='axes fraction', horizontalalignment='left', fontsize=18, color='C0')


                ax.set_xticks(np.arange(1980,2021,10))
                ax.set_xlim(1979,2022)
                ax.set(ylim=(0,8), yticks=np.arange(0, 8.1, 2))
                ax.tick_params(axis='x', direction='out',length=3,labelsize=16,pad=6)
                ax.tick_params(axis='y', direction='out',length=3,labelsize=16,pad=10)

                ax.annotate(f'#{node}', xy=(0.96, 0.87),xycoords='axes fraction',horizontalalignment='right',fontsize=22,color='black',bbox=dict(facecolor='lightgrey', edgecolor='None', alpha=0.8))

        plt.subplots_adjust(left=0.1, right=0.9, top=0.9, bottom=0.1)

        plt.show()
        fig3.savefig(in_path+f"gph_{grid_name}_total_{day}d_{threshold}_lde_wwe_change.png",dpi=1000,bbox_inches='tight')

For the seasonal analysis, please refer to the 3.1 in other codes and add the loops by yourself.